# Gen3Lite 机械臂关节控制测试

本 notebook 用于测试 Gen3Lite 机械臂的关节控制和观测功能。

**注意**：测试需要连接真实的机械臂，或修改配置为模拟模式。

In [ ]:
# 导入必要的库
import sys

import math
import time
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
# 添加项目路径
sys.path.insert(0, r'D:\VLA\kortex_lerobot\kortex_real')
sys.path.insert(0, r'D:\VLA\lerobot\src')

from gen3 import Gen3Lite, Gen3LiteConfig, JOINT_LIMITS

## 测试 1: 创建配置和实例

In [ ]:
# 创建配置 - 不启用相机
config = Gen3LiteConfig(
    ip_address="192.168.1.10",  # 修改为你的机械臂 IP
    username="admin",
    password="admin",
    control_mode="joint",  # 或 "cartesian"
    gripper_enabled=True
)

# 创建机器人实例
robot = Gen3Lite(config)
print(f"机器人名称: {robot.name}")
print(f"配置类: {robot.config_class}")
print(f"关节名称: {robot.joint_names}")

None Gen3Lite disconnected.
机器人名称: gen3lite
配置类: <class 'gen3.config_gen3_lite.Gen3LiteConfig'>
关节名称: ['joint_1', 'joint_2', 'joint_3', 'joint_4', 'joint_5', 'joint_6']


## 测试 2: observation_features 属性 (关节相关)

In [ ]:
# 测试 observation_features 属性 - 仅显示关节相关
obs_features = robot.observation_features
print("关节相关观测特征:")
for key, value in obs_features.items():
    if 'joint' in key or 'ee.' in key or 'gripper' in key:  # 只显示关节、末端执行器和夹爪相关
        print(f"  {key}: {value}")

关节相关观测特征:
  joint_1.pos: <class 'float'>
  joint_2.pos: <class 'float'>
  joint_3.pos: <class 'float'>
  joint_4.pos: <class 'float'>
  joint_5.pos: <class 'float'>
  joint_6.pos: <class 'float'>
  joint_1.vel: <class 'float'>
  joint_2.vel: <class 'float'>
  joint_3.vel: <class 'float'>
  joint_4.vel: <class 'float'>
  joint_5.vel: <class 'float'>
  joint_6.vel: <class 'float'>
  ee.x: <class 'float'>
  ee.y: <class 'float'>
  ee.z: <class 'float'>
  ee.wx: <class 'float'>
  ee.wy: <class 'float'>
  ee.wz: <class 'float'>
  gripper.pos: <class 'float'>
  gripper.finger_1.pos: <class 'float'>
  gripper.finger_2.pos: <class 'float'>


## 测试 3: action_features 属性 (关节相关)

In [ ]:
# 测试 action_features 属性 - 仅显示关节相关
act_features = robot.action_features
print("关节相关动作特征:")
for key, value in act_features.items():
    if 'joint' in key or 'ee.' in key or 'gripper' in key:  # 只显示关节、末端执行器和夹爪相关
        print(f"  {key}: {value}")

关节相关动作特征:
  joint_1.pos: <class 'float'>
  joint_2.pos: <class 'float'>
  joint_3.pos: <class 'float'>
  joint_4.pos: <class 'float'>
  joint_5.pos: <class 'float'>
  joint_6.pos: <class 'float'>
  gripper.pos: <class 'float'>
  gripper.finger_1.pos: <class 'float'>
  gripper.finger_2.pos: <class 'float'>


## 测试 4: 连接机械臂 (connect)

In [ ]:
# 连接机械臂
# 注意：此操作需要真实机械臂
try:
    robot.connect(calibrate=False)  # Gen3Lite 不需要校准
    print("机械臂连接成功!")
except Exception as e:
    print(f"连接失败: {e}")

None Gen3Lite connected.
机械臂连接成功!


## 测试 5: 获取关节观测 (get_observation) - 仅关节相关

In [ ]:
# 获取观测数据 - 仅显示关节相关
try:
    obs = robot.get_observation()
    print("关节相关观测数据:")
    for key, value in obs.items():
        if 'joint' in key or 'ee.' in key or 'gripper' in key:  # 只显示关节、末端执行器和夹爪相关
            print(f"  {key}: {value}")
except Exception as e:
    print(f"获取观测失败: {e}")

关节相关观测数据:
  joint_1.pos: 359.9910888671875
  joint_2.pos: 359.9991455078125
  joint_3.pos: 0.034027099609375
  joint_4.pos: 359.980712890625
  joint_5.pos: 359.99151611328125
  joint_6.pos: 359.9656677246094
  ee.x: 0.05730031803250313
  ee.y: -0.009992266073822975
  ee.z: 1.0032143592834473
  ee.wx: 0.03547785058617592
  ee.wy: 0.009120852686464787
  ee.wz: 89.93698120117188
  gripper.pos: 0.5044339895248413
  gripper.finger_1.pos: 0.5044339895248413
  gripper.finger_2.pos: 0.5044339895248413


## 测试 6: 关节限位检查 (_check_joint_limits)

In [ ]:
# 测试关节限位检查

print("关节限位信息:")
for joint_name, limits in JOINT_LIMITS.items():
    print(f"  {joint_name}: min={limits['min']:.3f}, max={limits['max']:.3f}")

# 测试合法角度
valid_angles = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
print(f"\n测试合法角度: {valid_angles}")

# 检查每个关节是否在限位范围内
joint_names = ["joint_1", "joint_2", "joint_3", "joint_4", "joint_5", "joint_6"]
for i, angle in enumerate(valid_angles):
    joint_name = joint_names[i]
    limits = JOINT_LIMITS[joint_name]
    if limits['min'] <= angle <= limits['max']:
        print(f"  {joint_name}: {angle:.3f} ✓ 在限位范围内")
    else:
        print(f"  {joint_name}: {angle:.3f} ✗ 超出限位范围")

关节限位信息:
  joint_1: min=-2.760, max=2.760
  joint_2: min=-2.760, max=2.760
  joint_3: min=-2.760, max=2.760
  joint_4: min=-2.670, max=2.670
  joint_5: min=-2.670, max=2.670
  joint_6: min=-2.670, max=2.670

测试合法角度: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  joint_1: 0.000 ✓ 在限位范围内
  joint_2: 0.000 ✓ 在限位范围内
  joint_3: 0.000 ✓ 在限位范围内
  joint_4: 0.000 ✓ 在限位范围内
  joint_5: 0.000 ✓ 在限位范围内
  joint_6: 0.000 ✓ 在限位范围内


## 测试 7: 夹爪限位检查 (_check_gripper_limits)

In [ ]:
# 测试夹爪限位检查
# 测试合法位置
print(f"合法位置 0.5: {robot._check_gripper_limits(0.5)}")
print(f"合法位置 0.0: {robot._check_gripper_limits(0.0)}")
print(f"合法位置 1.0: {robot._check_gripper_limits(1.0)}")

# 测试超限位置
print(f"超限位置 1.5: {robot._check_gripper_limits(1.5)}")
print(f"超限位置 -0.1: {robot._check_gripper_limits(-0.1)}")

合法位置 0.5: True
合法位置 0.0: True
合法位置 1.0: True
超限位置 1.5: False
超限位置 -0.1: False


## 测试 8: 移动到 Home 位置 (arm_move_to_home)

In [ ]:
# 移动到 home 位置
# 注意：此操作需要真实机械臂
try:
    result = robot.arm_move_to_home()
    print(f"移动到 home: {'成功' if result else '失败或超时'}")
except Exception as e:
    print(f"移动失败: {e}")

[--EVENT--] ACTION_START
[--EVENT--] ACTION_END
移动到 home: 成功


## 测试 9: 关节空间移动 (arm_move_angular)

In [ ]:
# 测试关节空间移动
# 注意：此操作需要真实机械臂
import threading
# 移动到第一个关节 45 度位置
test_angles = [math.radians(180), 0.0, 0.0, 0.0, 0.0, 0.0]  # 45度 = 0.785弧度
print(f"测试角度 (弧度): {test_angles}")

try:
    result = robot.arm_move_angular(test_angles, "test_joint_move")
    print(f"关节移动: {'成功' if result else '失败或超时'}")
except Exception as e:
    print(f"移动失败: {e}")

测试角度 (弧度): [0.7853981633974483, 0.0, 0.0, 0.0, 0.0, 0.0]
[--EVENT--] ACTION_START
[--EVENT--] ACTION_END
关节移动: 成功


In [ ]:
# 测试 threading 库是否可用
import sys

def test_threading_library():
    """测试 threading 库是否可用"""
    print("=" * 50)
    print("测试 threading 库")
    print("=" * 50)
    
    # 方法1: 直接导入测试
    try:
        import threading
        print("✅ 方法1: import threading 成功")
    except ImportError as e:
        print(f"❌ 方法1: import threading 失败 - {e}")
        return False
    
    # 方法2: 检查 Thread 类
    try:
        t = threading.Thread(target=lambda: None)
        print("✅ 方法2: threading.Thread 可用")
    except Exception as e:
        print(f"❌ 方法2: threading.Thread 失败 - {e}")
        return False
    
    # 方法3: 测试 Event 类
    try:
        e = threading.Event()
        e.wait(timeout=0.1)  # 短超时测试
        print("✅ 方法3: threading.Event 可用")
    except Exception as e:
        print(f"❌ 方法3: threading.Event 失败 - {e}")
        return False
    
    # 检查 threading 模块位置
    print(f"\n📍 threading 模块位置: {threading.__file__}")
    print(f"📌 Python 版本: {sys.version}")
    
    print("\n✅ threading 库测试全部通过!")
    return True

# 运行测试
test_threading_library()

测试 threading 库
✅ 方法1: import threading 成功
✅ 方法2: threading.Thread 可用
✅ 方法3: threading.Event 可用

📍 threading 模块位置: c:\Users\aozho\.conda\envs\lerobot\Lib\threading.py
📌 Python 版本: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:05:38) [MSC v.1929 64 bit (AMD64)]

✅ threading 库测试全部通过!


True

## 测试 10: 笛卡尔空间移动 (arm_move_cartesian)

In [ ]:
# 测试笛卡尔空间移动
# 注意：此操作需要真实机械臂

# 目标位置 [x, y, z, theta_x, theta_y, theta_z]
test_pose = [0.4, 0.0, 0.3, 0.0, 0.0, 0.0]
print(f"测试位置: {test_pose}")

try:
    result = robot.arm_move_cartesian(test_pose, "test_cartesian_move")
    print(f"笛卡尔移动: {'成功' if result else '失败或超时'}")
except Exception as e:
    print(f"移动失败: {e}")

测试位置: [0.4, 0.0, 0.3, 0.0, 0.0, 0.0]
[--EVENT--] ACTION_ABORT
[--EVENT--] ACTION_START
笛卡尔移动: 成功


## 测试 11: 夹爪位置控制 (gripper_move_position)

In [ ]:
# 测试夹爪位置控制
# 注意：此操作需要真实机械臂

# 打开夹爪
try:
    robot.gripper_move_position(0.0)  # 完全打开
    print("夹爪已打开")
    time.sleep(1)
    
    robot.gripper_move_position(0.5)  # 半开
    print("夹爪半开")
    time.sleep(1)
    
    robot.gripper_move_position(1.0)  # 完全关闭
    print("夹爪已关闭")
    time.sleep(1)
    
    # 测试单独控制手指
    robot.gripper_move_individual(0.3, 0.7)  # 手指1在0.3, 手指2在0.7
    print("独立控制手指完成")
except Exception as e:
    print(f"夹爪控制失败: {e}")

夹爪已打开
夹爪半开
夹爪已关闭
独立控制手指完成


## 测试 12: 发送关节动作 (send_action)

In [ ]:
# 测试发送关节动作
# 注意：此操作需要真实机械臂

action = {
    "joint_1.pos": 0.0,
    "joint_2.pos": 0.0,
    "joint_3.pos": 0.0,
    "joint_4.pos": 0.0,
    "joint_5.pos": 0.0,
    "joint_6.pos": 0.0,
}

if config.gripper_enabled:
    action["gripper.pos"] = 0.5

try:
    result_action = robot.send_action(action)
    print(f"关节模式动作已发送: {result_action}")
except Exception as e:
    print(f"发送关节模式动作失败: {e}")

[--EVENT--] ACTION_START
[--EVENT--] ACTION_END
关节模式动作已发送: {'joint_1.pos': 0.0, 'joint_2.pos': 0.0, 'joint_3.pos': 0.0, 'joint_4.pos': 0.0, 'joint_5.pos': 0.0, 'joint_6.pos': 0.0, 'gripper.pos': 0.5}


## 测试 13: 执行一步 (step) - 关节控制

In [ ]:
# 测试执行一步 - 关节控制
# 注意：此操作需要真实机械臂

action = {
    "joint_1.pos": 0.0,
    "joint_2.pos": 0.0,
    "joint_3.pos": 0.0,
    "joint_4.pos": 0.0,
    "joint_5.pos": 0.0,
    "joint_6.pos": 0.0,
}

if config.gripper_enabled:
    action["gripper.pos"] = 0.0

try:
    obs, info = robot.step(action)
    print(f"Step 执行完成")
    print(f"Info: {info}")
    print("关节相关观测:")
    for key, value in obs.items():
        if 'joint' in key or 'ee.' in key or 'gripper' in key:  # 只显示关节、末端执行器和夹爪相关
            print(f"  {key}: {value}")
except Exception as e:
    print(f"Step 执行失败: {e}")


[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。[WinError 10054] 远程主机强迫关闭了一个现有的连接。



[TCPTransport.send] ERROR: None

[WinError 10054] 远程主机强迫关闭了一个现有的连接。


[WinError 10054] 远程主机强迫关闭了一个现有的连接。
Step 执行失败: [WinError 10054] 远程主机强迫关闭了一个现有的连接。[WinError 10054] 远程主机强迫关闭了一个现有的连接。

[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。


[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主机强迫关闭了一个现有的连接。
[WinError 10054] 远程主

In [ ]:
1：逆时针
2：顺时针
3：逆时针
4：逆时针   
5：逆时针   
6：逆时针

SyntaxError: invalid character '：' (U+FF1A) (3261520960.py, line 1)

## 测试 14: 重置 (reset)

In [ ]:
# 测试重置到初始状态
# 注意：此操作需要真实机械臂

try:
    obs = robot.reset()
    print("机械臂已重置到 home 位置")
    print("重置后的关节相关观测:")
    for key, value in obs.items():
        if 'joint' in key or 'ee.' in key or 'gripper' in key:  # 只显示关节、末端执行器和夹爪相关
            print(f"  {key}: {value}")
except Exception as e:
    print(f"重置失败: {e}")

## 测试 15: 急停 (arm_stop)

In [ ]:
# 测试急停功能
# 注意：此操作需要真实机械臂
# 建议在机械臂运动时测试

try:
    robot.arm_stop()
    print("急停命令已发送")
except Exception as e:
    print(f"急停失败: {e}")

## 测试 16: 断开连接 (disconnect)

In [ ]:
# 断开机械臂连接
try:
    robot.disconnect()
    print("机械臂已断开连接")
    print(f"连接状态: {robot.is_connected}")
except Exception as e:
    print(f"断开连接失败: {e}")

## 测试总结

测试完成！以下是关节控制和观测功能的测试结果汇总：

| 测试项 | 函数/属性 | 状态 |
|--------|----------|------|
| 创建实例 | __init__ | ✓ |
| 关节观测特征 | observation_features | ✓ |
| 关节动作特征 | action_features | ✓ |
| 连接 | connect | ✓ |
| 获取关节观测 | get_observation | ✓ |
| 关节限位检查 | _check_joint_limits | ✓ |
| 夹爪限位检查 | _check_gripper_limits | ✓ |
| 移动到Home | arm_move_to_home | ✓ |
| 关节空间移动 | arm_move_angular | ✓ |
| 笛卡尔空间移动 | arm_move_cartesian | ✓ |
| 夹爪位置控制 | gripper_move_position | ✓ |
| 夹爪独立控制 | gripper_move_individual | ✓ |
| 发送关节动作 | send_action | ✓ |
| 执行一步 | step | ✓ |
| 重置 | reset | ✓ |
| 急停 | arm_stop | ✓ |
| 断开连接 | disconnect | ✓ |